In [9]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

In [12]:
from os import name


(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

# 구조 변경(차원)
print(x_train.shape)    # (60000,, 28, 28)
x_train = x_train.reshape((-1, 28, 28, 1)).astype('float32') / 255.0
x_test = x_test.reshape((-1, 28, 28, 1)).astype('float32') / 255.0
print(x_train.shape)    # (60000, 28, 28, 1)

# 모델 정의
inputs = tf.keras.layers.Input(shape=(28, 28, 1))

'''
# 방법 1
x = tf.keras.layers.Conv2D(filters=16, kernel_size=(3,3), padding='same', activation='relu')(inputs)
x = tf.keras.layers.MaxPool2D(pool_size=(2, 2))(x)
x = tf.keras.layers.Dropout(rate=0.2)(x)

x = tf.keras.layers.Conv2D(filters=32, kernel_size=(3,3), padding='same', activation='relu')(x)
x = tf.keras.layers.MaxPool2D(pool_size=(2, 2))(x)

x = tf.keras.layers.Conv2D(filters=32, kernel_size=(3,3), padding='same', activation='relu')(x)
x = tf.keras.layers.MaxPool2D(pool_size=(2, 2))(x)

# Fully Connected Layers
x = tf.keras.layers.Flatten()(x)
x = tf.keras.layers.Dense(units=64, activation='relu')(x)
x = tf.keras.layers.Dropout(rate=0.3)(x)
x = tf.keras.layers.Dense(units=32, activation='relu')(x)
x = tf.keras.layers.Dropout(rate=0.2)(x)
outputs = tf.keras.layers.Dense(units=10, activation='softmax')(x)
'''

# 방법2 - BatchNormation: Conv/Dense 뒤에 베치 - 학습 안정화, 수련 가속화
# use_bias=False: Conv/Dense의 bias 제거
x = tf.keras.layers.Conv2D(16, (3,3), padding='same', use_bias=False)(inputs)
x = tf.keras.layers.BatchNormalization()(x)
x = tf.keras.layers.ReLU()(x)
x = tf.keras.layers.MaxPool2D((2, 2))(x)
x = tf.keras.layers.Dropout(rate=0.25)(x)

x = tf.keras.layers.Conv2D(16, (3,3), padding='same', use_bias=False)(x)
x = tf.keras.layers.BatchNormalization()(x)
x = tf.keras.layers.ReLU()(x)
x = tf.keras.layers.MaxPool2D((2, 2))(x)
x = tf.keras.layers.Dropout(rate=0.25)(x)

x = tf.keras.layers.Conv2D(16, (3,3), padding='same', use_bias=False)(x)
x = tf.keras.layers.BatchNormalization()(x)
x = tf.keras.layers.ReLU()(x)
x = tf.keras.layers.MaxPool2D((2, 2))(x)
x = tf.keras.layers.Dropout(rate=0.25)(x)

# Fully Connected Layers
x = tf.keras.layers.Flatten()(x)
x = tf.keras.layers.Dense(64, use_bias=False)(x)
x = tf.keras.layers.BatchNormalization()(x)
x = tf.keras.layers.ReLU()(x)
x = tf.keras.layers.Dropout(0.3)(x)

x = tf.keras.layers.Dense(32)(x)
x = tf.keras.layers.BatchNormalization()(x)
x = tf.keras.layers.ReLU()(x)
x = tf.keras.layers.Dropout(0.3)(x)

outputs = tf.keras.layers.Dense(10, activation='softmax')(x)


# 모델 객체 생성
model = tf.keras.models.Model(inputs=inputs, outputs=outputs, name='mnist_cnn_func')

print(model.summary())

(60000, 28, 28)
(60000, 28, 28, 1)
Model: "mnist_cnn_func"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_8 (InputLayer)        [(None, 28, 28, 1)]       0         
                                                                 
 conv2d_19 (Conv2D)          (None, 28, 28, 16)        144       
                                                                 
 batch_normalization_6 (Batc  (None, 28, 28, 16)       64        
 hNormalization)                                                 
                                                                 
 re_lu_6 (ReLU)              (None, 28, 28, 16)        0         
                                                                 
 max_pooling2d_19 (MaxPoolin  (None, 14, 14, 16)       0         
 g2D)                                                            
                                                                 
 dropout_19 (Drop

In [ ]:
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
es = tf.keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True)

history = model.fit(
    x_train, y_train, epochs=100, batch_size=128, validation_split=0.1, callbacks=[es], verbose=2
)

# 모델 평가
train_loss, train_acc = model.evaluate(x_train, y_train, verbose=0)
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
print(f'train_loss:{train_loss:.4f}, train_acc:{train_acc:.4f}')
print(f'test_loss:{train_loss:.4f}, test_acc:{train_acc:.4f}')

Epoch 1/100
422/422 - 5s - loss: 0.7026 - accuracy: 0.7663 - val_loss: 0.0962 - val_accuracy: 0.9710 - 5s/epoch - 13ms/step
Epoch 2/100
422/422 - 1s - loss: 0.2129 - accuracy: 0.9379 - val_loss: 0.0670 - val_accuracy: 0.9805 - 1s/epoch - 3ms/step
Epoch 3/100
422/422 - 1s - loss: 0.1534 - accuracy: 0.9562 - val_loss: 0.0534 - val_accuracy: 0.9842 - 1s/epoch - 3ms/step
Epoch 4/100
422/422 - 1s - loss: 0.1228 - accuracy: 0.9650 - val_loss: 0.0564 - val_accuracy: 0.9832 - 1s/epoch - 3ms/step
Epoch 5/100
422/422 - 1s - loss: 0.1060 - accuracy: 0.9706 - val_loss: 0.0523 - val_accuracy: 0.9843 - 1s/epoch - 3ms/step
Epoch 6/100
422/422 - 1s - loss: 0.0937 - accuracy: 0.9742 - val_loss: 0.0439 - val_accuracy: 0.9868 - 1s/epoch - 3ms/step
Epoch 7/100
422/422 - 1s - loss: 0.0838 - accuracy: 0.9761 - val_loss: 0.0399 - val_accuracy: 0.9882 - 1s/epoch - 3ms/step
Epoch 8/100
422/422 - 1s - loss: 0.0763 - accuracy: 0.9794 - val_loss: 0.0437 - val_accuracy: 0.9878 - 1s/epoch - 3ms/step
Epoch 9/100
422